# Bradley-Terry ratings for SEA-HELM

Fits [`src/elo`](../../src/elo/) Bradley-Terry ratings to head-to-head contests built by
[`bt_dataloader.py`](../../src/elo/bt_dataloader.py).

`load()` caches the run-averaged scores, so the read happens **once**. Retuning `tolerance`,
switching view, changing the contest budget, and every bootstrap replicate all work from the 
cache and never touch the results tree again.

Sampling is the whole cost of a `build()` — the fit itself is milliseconds — and it is
proportional to the contests drawn rather than to the cells' `(item, pair)` grid, so a
2,000,000-contest view costs about two seconds rather than a couple of minutes. Bootstrap
replicates are independent draws, so `num_proceses` spends cores on them without changing a rating.

What the loader does, for reference when reading the numbers below:

- each item's score is the mean over the runs that scored it;
- an item is used in a cell only if **every** competitor in that cell has it — a model missing
  a file or an item does not compete there, and is never scored zero;
- scores are divided by `METRIC_NATIVE_MAX` before the tie tolerance applies, so one tolerance
  means the same thing on different metrics;
- the contest budget is split with an equal share per level (language → competency → task), with
  an aggregation group counting as **one** task.

## Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd

# The loader lives in `src.elo`, so the repo root needs to be importable regardless of where
# the kernel started.
REPO_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "src" / "aggregation" / "constants.py").exists()
)
LOADER_DIR = REPO_ROOT / "helpers" / "elo"  # cache and notebook outputs stay here
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.elo.bt_dataloader import BTDataLoader  # noqa: E402

print("repo root:", REPO_ROOT)

## Configuration

In [ ]:
# set SWEEP_DIRS with model types, eg
# {"<full path to instruct-folder": "instruct", "<full path to reasoning-folder": "reasoning"}
# Each key is the path to a sweep directory holding "<org>/<leaf>/run_<r>".
SWEEP_DIRS = {}

N_RUNS = 8
TOLERANCE = 0.01  # on the METRIC_NATIVE_MAX-rescaled [0, 1] scale
N_CONTESTS = 2_000_000
SEED = 94370244
NUM_PROCESSES = 32

CACHE_DIR = str(LOADER_DIR / ".bt_cache")
OUTPUT_DIR = str(REPO_ROOT / "results" / "elo-bt")

SIGNIFICANCE = 0.05  # 0.05 -> 95% intervals
N_BOOTSTRAP = 2000  # item-level replicates
KEEP_TIES = False  # whether to keep ties in the BT model (or drop them)
TIE_BUDGET = "decisive"  # "decisive" or "inflate" -- how to handle ties in the BT model (if `KEEP_TIES` is True)

# "pool" or "equal" -- how to allocate contests in the BT model. "equal" means that each pair
# of items gets the same number of contests, "pool" means that the total number of contests is
# fixed and allocated according to the empirical distribution of pairs in the data.
PAIR_BUDGET = "equal"

LOADER_KWARGS = {
    "sweep_dirs": SWEEP_DIRS,
    "n_runs": N_RUNS,
    "tolerance": TOLERANCE,
    "n_contests": N_CONTESTS,
    "seed": SEED,
    "num_processes": NUM_PROCESSES,
    "cache_dir": CACHE_DIR,
    "output_dir": OUTPUT_DIR,
    "keep_ties": KEEP_TIES,
    "tie_budget": TIE_BUDGET,
    "pair_budget": PAIR_BUDGET,
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print("outputs:", OUTPUT_DIR)

## Load — the expensive stage

First call reads the tree; every later call with the same `LOADER_KWARGS` restores the cache in
well under a second. Pass `use_cache=False` to force a re-read.

In [ ]:
started = time.time()
loader = BTDataLoader(**LOADER_KWARGS).load()
elapsed = time.time() - started

print(f"load {elapsed:.1f}s")
print(f"{len(loader.models)} competitors, {len(loader.cells)} (task, language) cells")
print(f"{len(loader.failed_loads)} failed file reads")

## Calculate Elo rating andConfidence intervals

In [ ]:
def summarise(replicates, significance=SIGNIFICANCE):
    """Percentile intervals over the bootstrap replicates."""
    return pd.DataFrame(
        {
            "rating": replicates.mean(),
            "ci_lower": replicates.quantile(significance / 2),
            "ci_upper": replicates.quantile(1 - significance / 2),
            "std": replicates.std(ddof=1),
            "n_replicates": replicates.notna().sum(),
        }
    ).rename_axis("models")


# `loader.bootstrap_ratings` draws and fits each replicate and returns one row per replicate,
# indexed by competitor *name*: a `PairDataset`'s competitors come from the rows actually sampled.
started = time.time()
item_replicates = loader.bootstrap_ratings(
    n_replicates=N_BOOTSTRAP,
    bootstrap="items",
    category="sea",
    balance_by="language",
)
print(f"{N_BOOTSTRAP} item-level replicates in {time.time() - started:.1f}s")

board = summarise(item_replicates)
board = board.sort_values("rating", ascending=False)

model = board.index.to_series().str.split("/", n=1, expand=True)
board.insert(0, "model", model[1])
board.insert(1, "model_type", model[0])
board.to_csv(Path(OUTPUT_DIR) / "sea_elo_ratings.csv", index=False)

In [ ]:
SEA_LANGUAGES = ["id", "ms", "th", "vi", "tl", "my", "ta"]
for category, label, balance_by in [
    ["language", lang, "language"] for lang in SEA_LANGUAGES
]:
    started = time.time()
    LOADER_KWARGS["n_contests"] = (
        500_000  # reduce n contests for language-level bootstraps
    )
    loader = BTDataLoader(**LOADER_KWARGS).load()
    item_replicates = loader.bootstrap_ratings(
        n_replicates=N_BOOTSTRAP,
        bootstrap="items",
        category=category,
        label=label,
        balance_by=balance_by,
    )
    print(f"{N_BOOTSTRAP} item-level replicates in {time.time() - started:.1f}s")

    board = summarise(item_replicates)
    board = board.sort_values("rating", ascending=False)

    model = board.index.to_series().str.split("/", n=1, expand=True)
    board.insert(0, "model", model[1])
    board.insert(1, "model_type", model[0])
    board.to_csv(Path(OUTPUT_DIR) / f"{label}_elo_ratings.csv", index=False)